# Customer Groups: Complexity & Profitability Analysis

**Business Question:** Which customer groups are most profitable and easy to serve? Which are costly to support?

**Context:** This notebook computes profitability and complexity scores for customer groups from raw silver-layer transaction data, enabling interactive exploration of the customer portfolio matrix.

**Calculation Method:** Aggregates by Customer Group, then applies deterministic percentile-ranking formulas per [spec.md](../demos/charts/chart-customer_matrix/spec.md).

**Output:** 
- Interactive DataFrames with filtering and summary stats
- Scatter plot visualization (complexity vs. profitability)
- CSV and XLSX exports of results

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Color palette (Helbling style)
BLUE = "#1f4e79"
LBLUE = "#5b9bd5"
ORANGE = "#ed7d31"
GREY = "#a6a6a6"
GREEN = "#2e7d32"
RED = "#c0392b"
LIGHT_GREEN = "#e2efda"
LIGHT_RED = "#f8dcdb"
LIGHT_YELLOW = "#fff2cc"
LIGHT_BLUE = "#e7f0f7"

# Paths
NOTEBOOK_DIR = Path(os.getcwd())
REPO = NOTEBOOK_DIR.parent
SILVER_DIR = REPO / 'data' / 'silver'
CUSTOMER_SILVER = SILVER_DIR / 'customer.parquet'
EXPORT_DIR = NOTEBOOK_DIR

print(f"Repository: {REPO}")
print(f"Silver data: {SILVER_DIR}")
print(f"Export to: {EXPORT_DIR}")

Repository: d:\GIT\GF-handout\gf-complexity-navigator
Silver data: d:\GIT\GF-handout\gf-complexity-navigator\data\silver
Export to: d:\GIT\GF-handout\gf-complexity-navigator\notebooks


## 2. Load Data from Silver Layer

In [2]:
# Load customer transaction data from silver layer
if not CUSTOMER_SILVER.exists():
    raise FileNotFoundError(f"Silver layer data not found: {CUSTOMER_SILVER}\nRun: python src/build_silver.py")

print(f"Loading from silver layer: {CUSTOMER_SILVER}")
customer_data = pd.read_parquet(CUSTOMER_SILVER)

print(f"\nLoaded: {len(customer_data):,} transaction rows")
print(f"Shape: {customer_data.shape}")
print(f"\nColumns: {customer_data.columns.tolist()}")
print(f"\nData types:")
print(customer_data.dtypes)

Loading from silver layer: d:\GIT\GF-handout\gf-complexity-navigator\data\silver\customer.parquet

Loaded: 115,824 transaction rows
Shape: (115824, 26)

Columns: ['year', 'month', 'buying_group_l6', 'customer_group', 'customer_number', 'customer_name', 'booked_date', 'sales_order_number', 'net_ordered_amount_lc', 'ns_ordered_amount_lc', 'business_unit', 'region', 'sub_region', 'sales_unit', 'local_currency', 'net_sales_lc', 'consolidated_gross_profit_lc_incl_freight', 'consolidated_gross_profit_lc', '_source_file', '_loaded_at', 'customer_id', 'is_negative_gp', 'is_zero_sales', 'is_cost_without_sales', 'is_missing_currency', 'is_missing_buying_group']

Data types:
year                                                  int64
month                                                 int64
buying_group_l6                                         str
customer_group                                          str
customer_number                                         str
customer_name              

## 3. Data Inspection & Preparation

In [3]:
# Basic stats
print(f"Total transaction rows: {len(customer_data):,}")
print(f"Unique customer groups: {customer_data['customer_group'].nunique()}")
print(f"Unique sales order numbers: {customer_data['sales_order_number'].nunique()}")
print(f"Date range: {customer_data['booked_date'].min()} to {customer_data['booked_date'].max()}")

print(f"\nKey columns for matrix calculation:")
print(f"  - customer_group: customer segment identifier")
print(f"  - net_sales_lc: net sales in local currency")
print(f"  - consolidated_gross_profit_lc: gross profit in local currency")
print(f"  - sales_order_number: order identifier (for counting orders)")
print(f"  - region: geographic region")
print(f"  - sales_unit: sales organizational unit")

print(f"\nMissing values:")
print(customer_data.isnull().sum())

# Add flag for negative-profit lines
customer_data['neg_line'] = (customer_data['consolidated_gross_profit_lc'] < 0).astype(int)
print(f"\nNegative-profit lines: {customer_data['neg_line'].sum():,} ({customer_data['neg_line'].mean()*100:.1f}%)")

Total transaction rows: 115,824
Unique customer groups: 2366
Unique sales order numbers: 103785
Date range: 1901-01-01 00:00:00 to 2026-05-05 00:00:00

Key columns for matrix calculation:
  - customer_group: customer segment identifier
  - net_sales_lc: net sales in local currency
  - consolidated_gross_profit_lc: gross profit in local currency
  - sales_order_number: order identifier (for counting orders)
  - region: geographic region
  - sales_unit: sales organizational unit

Missing values:
year                                             0
month                                            0
buying_group_l6                                  0
customer_group                                   0
customer_number                                  0
customer_name                                    0
booked_date                                  23252
sales_order_number                               0
net_ordered_amount_lc                            0
ns_ordered_amount_lc                      

## 4. Aggregate by Customer Group

In [4]:
# Group by customer group and compute aggregates
cg = customer_data.groupby('customer_group')

customer_matrix = pd.DataFrame({
    # Profitability drivers
    'net_sales': cg['net_sales_lc'].sum(),
    'gross_profit': cg['consolidated_gross_profit_lc'].sum(),
    # Complexity drivers
    'orders': cg['sales_order_number'].nunique(),
    'lines': cg.size(),
    'neg_line_share': cg['neg_line'].mean(),
    'regions': cg['region'].nunique(),
    'sales_units': cg['sales_unit'].nunique(),
}).reset_index()

# Derived metrics
customer_matrix['avg_order_value'] = np.where(
    customer_matrix['orders'] != 0,
    customer_matrix['net_sales'] / customer_matrix['orders'],
    0
)
customer_matrix['gp_pct'] = np.where(
    customer_matrix['net_sales'] != 0,
    customer_matrix['gross_profit'] / customer_matrix['net_sales'] * 100,
    0
)
customer_matrix['fragmentation'] = customer_matrix['regions'] + customer_matrix['sales_units']

print(f"Aggregated to {len(customer_matrix)} customer groups")
print(f"\nTop 10 by net_sales:")
print(customer_matrix.nlargest(10, 'net_sales')[['customer_group', 'net_sales', 'gross_profit', 'orders']].to_string())

Aggregated to 2366 customer groups

Top 10 by net_sales:
       customer_group     net_sales  gross_profit  orders
606                GC  1.072299e+08  5.880533e+07   45613
56    ALPHA-RFR-GROUP  4.184660e+07  2.330823e+07   16293
2126              VGH  3.036994e+07  1.595195e+07    1440
1903          SONEPAR  1.391420e+07  9.100579e+06    1565
1901            SOLAR  1.329114e+07  7.971148e+06     778
433               EDT  9.479942e+06  5.717621e+06    2764
1602     PFEIFFER&MAY  8.401927e+06  5.274011e+06    3360
691               GSH  5.354182e+06  3.459986e+06    3186
2291              WIM  4.878231e+06  2.825616e+06    2199
1512         NL-WASCO  3.941642e+06  2.282517e+06     714


## 5. Compute Profitability Score (Y-axis)

**Formula (per FR-009):**
- `value_raw = pct_rank(gross_profit) × 0.6 + pct_rank(net_sales) × 0.4`
- `profitability_pct = pct_rank(value_raw)` → 0-100 percentile

In [5]:
def pct_rank(s):
    """Percentile rank 0-100; NaNs ignored. Deterministic (rank method='average' for ties)."""
    return s.rank(pct=True, method="average") * 100.0

# Compute profitability score
customer_matrix['value_raw'] = (
    pct_rank(customer_matrix['gross_profit']) * 0.6
    + pct_rank(customer_matrix['net_sales']) * 0.4
)
customer_matrix['profitability_pct'] = pct_rank(customer_matrix['value_raw'])

print(f"Profitability Score Computed")
print(f"\nRange: {customer_matrix['profitability_pct'].min():.1f} - {customer_matrix['profitability_pct'].max():.1f}")
print(f"Mean: {customer_matrix['profitability_pct'].mean():.1f}")
print(f"\nTop 5 most profitable customer groups:")
print(customer_matrix.nlargest(5, 'profitability_pct')[['customer_group', 'gross_profit', 'net_sales', 'profitability_pct']].to_string())

Profitability Score Computed

Range: 0.0 - 100.0
Mean: 50.0

Top 5 most profitable customer groups:
       customer_group  gross_profit     net_sales  profitability_pct
606                GC  5.880533e+07  1.072299e+08         100.000000
56    ALPHA-RFR-GROUP  2.330823e+07  4.184660e+07          99.957735
2126              VGH  1.595195e+07  3.036994e+07          99.915469
1903          SONEPAR  9.100579e+06  1.391420e+07          99.873204
1901            SOLAR  7.971148e+06  1.329114e+07          99.830938


## 6. Compute Complexity Score (X-axis)

**Formula (per FR-010):** Four independent proxies, each percentile-ranked and weighted:
- **Order frequency** (30%): `pct_rank(orders)`
- **Small order burden** (30%): `pct_rank(-avg_order_value)` [negative: small AOV = high complexity]
- **Margin leakage** (20%): `pct_rank(neg_line_share)`
- **Fragmentation** (20%): `pct_rank(regions + sales_units)`

Then: `complexity_pct = pct_rank(weighted_sum)` → 0-100 percentile

In [8]:
# Compute four complexity proxies
proxy_order_freq = pct_rank(customer_matrix['orders'])
proxy_small_order = pct_rank(-customer_matrix['avg_order_value'])  # negative: small AOV = high complexity
proxy_margin_leak = pct_rank(customer_matrix['neg_line_share'])
proxy_fragmentation = pct_rank(customer_matrix['fragmentation'])

# Weighted combination
customer_matrix['complexity_raw'] = (
    proxy_order_freq * 0.30
    + proxy_small_order * 0.30
    + proxy_margin_leak * 0.20
    + proxy_fragmentation * 0.20
)
customer_matrix['complexity_pct'] = pct_rank(customer_matrix['complexity_raw'])

print(f"Complexity Score Computed")
print(f"\nRange: {customer_matrix['complexity_pct'].min():.1f} - {customer_matrix['complexity_pct'].max():.1f}")
print(f"Mean: {customer_matrix['complexity_pct'].mean():.1f}")
print(f"\nTop 5 most complex customer groups:")
print(customer_matrix.nlargest(5, 'complexity_pct')[['customer_group', 'orders', 'avg_order_value', 'complexity_pct']].to_string())

Complexity Score Computed

Range: 0.0 - 100.0
Mean: 50.0

Top 5 most complex customer groups:
               customer_group  orders  avg_order_value  complexity_pct
2057               TIM RÖNNAU      86        -0.532816      100.000000
364              DENNE HUBERT      33        -0.096970       99.957735
1698  RO-BI TEC GMBH & CO. KG      14      -136.428571       99.915469
1770    SANITÄRTECHNIK REISER      22       -60.390909       99.873204
2055               TIM BRÄKER      13       -92.307692       99.830938


## 7. Quadrant Classification

**Strategy Quadrants (per FR-011 to FR-013):** Based on 60/40 percentile thresholds:

| Profitability | Complexity | Quadrant | Strategy |
|---|---|---|---|
| ≥ 60 | < 40 | **Top-left** | Grow (Valuable, Easy) |
| ≥ 60 | ≥ 60 | **Top-right** | Protect / Serve Differently (Valuable, Costly) |
| < 40 | < 40 | **Bottom-left** | Simplify / Steer (Marginal, Easy) |
| < 40 | ≥ 60 | **Bottom-right** | Selective Deprioritization (Marginal, Costly) |
| Other | Other | **Center** | Mid-range |


In [9]:
def assign_quadrant(prof_pct, cplx_pct):
    """Assign customer group to quadrant based on 60/40 percentile thresholds."""
    if prof_pct >= 60 and cplx_pct < 40:
        return "Top-left (High Profit, Low Complexity)"
    elif prof_pct >= 60 and cplx_pct >= 60:
        return "Top-right (High Profit, High Complexity)"
    elif prof_pct < 40 and cplx_pct < 40:
        return "Bottom-left (Low Profit, Low Complexity)"
    elif prof_pct < 40 and cplx_pct >= 60:
        return "Bottom-right (Low Profit, High Complexity)"
    else:
        return "Center (Mid-range)"

customer_matrix['quadrant'] = [
    assign_quadrant(p, k)
    for p, k in zip(customer_matrix['profitability_pct'], customer_matrix['complexity_pct'])
]

print(f"Quadrant Classification Complete")
print(f"\nDistribution across quadrants:")
quad_dist = customer_matrix['quadrant'].value_counts()
for quad, count in quad_dist.items():
    pct = count / len(customer_matrix) * 100
    print(f"  {quad:50s} {count:3d} ({pct:5.1f}%)")

Quadrant Classification Complete

Distribution across quadrants:
  Center (Mid-range)                                 830 ( 35.1%)
  Bottom-right (Low Profit, High Complexity)         625 ( 26.4%)
  Top-left (High Profit, Low Complexity)             613 ( 25.9%)
  Top-right (High Profit, High Complexity)           196 (  8.3%)
  Bottom-left (Low Profit, Low Complexity)           102 (  4.3%)


## 8. Interactive Exploration

In [10]:
# Sort by net_sales descending for reference
customer_matrix = customer_matrix.sort_values('net_sales', ascending=False).reset_index(drop=True)

print("="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"\nTotal customer groups: {len(customer_matrix)}")
print(f"Total net sales: {customer_matrix['net_sales'].sum():,.0f}")
print(f"Total gross profit: {customer_matrix['gross_profit'].sum():,.0f}")
print(f"Blended margin: {customer_matrix['gross_profit'].sum() / customer_matrix['net_sales'].sum() * 100:.1f}%")

print(f"\nQuadrant Distribution:")
quad_dist = customer_matrix['quadrant'].value_counts()
for quad in ["Top-left (High Profit, Low Complexity)", "Top-right (High Profit, High Complexity)",
            "Bottom-left (Low Profit, Low Complexity)", "Bottom-right (Low Profit, High Complexity)", "Center (Mid-range)"]:
    count = quad_dist.get(quad, 0)
    pct = count / len(customer_matrix) * 100 if count > 0 else 0
    short_name = quad.split("(")[0].strip()
    print(f"  {short_name:20s} {count:3d} ({pct:5.1f}%)")

print(f"\nTop customer group per strategic quadrant (by net_sales):")
for quad in ["Top-left (High Profit, Low Complexity)", "Top-right (High Profit, High Complexity)",
            "Bottom-right (Low Profit, High Complexity)"]:
    quad_cg = customer_matrix[customer_matrix['quadrant'] == quad]
    if not quad_cg.empty:
        quad_top = quad_cg.iloc[0]
        short_name = quad.split("(")[0].strip()
        print(f"  {short_name:20s} {quad_top['customer_group']:40s} {quad_top['net_sales']:15,.0f}")

print("\n" + "="*80)

SUMMARY STATISTICS

Total customer groups: 2366
Total net sales: 268,805,873
Total gross profit: 147,248,717
Blended margin: 54.8%

Quadrant Distribution:
  Top-left             613 ( 25.9%)
  Top-right            196 (  8.3%)
  Bottom-left          102 (  4.3%)
  Bottom-right         625 ( 26.4%)
  Center               830 ( 35.1%)

Top customer group per strategic quadrant (by net_sales):
  Top-left             MvD Gasarmaturen                                 111,631
  Top-right            GC                                           107,229,884
  Bottom-right         Wilhelm Layher GmbH & Co KG                    3,180,677



### Top Customers by Net Sales

In [11]:
print("\nTop 15 customer groups by net_sales:")
display_cols = ['customer_group', 'net_sales', 'gross_profit', 'orders', 'avg_order_value', 'profitability_pct', 'complexity_pct', 'quadrant']
display(customer_matrix[display_cols].head(15))


Top 15 customer groups by net_sales:


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
0,GC,1.072299e+08,5.880533e+07,45613,2350.862339,100.000000,86.348267,"Top-right (High Profit, High Complexity)"
1,ALPHA-RFR-GROUP,4.184660e+07,2.330823e+07,16293,2568.379125,99.957735,86.432798,"Top-right (High Profit, High Complexity)"
2,VGH,3.036994e+07,1.595195e+07,1440,21090.234625,99.915469,84.995773,"Top-right (High Profit, High Complexity)"
3,SONEPAR,1.391420e+07,9.100579e+06,1565,8890.860557,99.873204,59.974641,Center (Mid-range)
4,SOLAR,1.329114e+07,7.971148e+06,778,17083.721137,99.830938,62.848690,"Top-right (High Profit, High Complexity)"
5,EDT,9.479942e+06,5.717621e+06,2764,3429.790968,99.788673,85.587489,"Top-right (High Profit, High Complexity)"
6,PFEIFFER&MAY,8.401927e+06,5.274011e+06,3360,2500.573548,99.746407,64.370245,"Top-right (High Profit, High Complexity)"
7,GSH,5.354182e+06,3.459986e+06,3186,1680.534259,99.704142,86.939983,"Top-right (High Profit, High Complexity)"
8,WIM,4.878231e+06,2.825616e+06,2199,2218.385931,99.661877,86.390533,"Top-right (High Profit, High Complexity)"
9,NL-WASCO,3.941642e+06,2.282517e+06,714,5520.506914,99.577346,61.158073,"Top-right (High Profit, High Complexity)"


### Top 3 per Quadrant

In [12]:
print("\nTop 3 per quadrant (by net_sales):")
for quad in customer_matrix['quadrant'].unique():
    quad_cg = customer_matrix[customer_matrix['quadrant'] == quad].nlargest(3, 'net_sales')
    print(f"\n{quad}:")
    display(quad_cg[display_cols])


Top 3 per quadrant (by net_sales):

Top-right (High Profit, High Complexity):


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
0,GC,1.072299e+08,5.880533e+07,45613,2350.862339,100.000000,86.348267,"Top-right (High Profit, High Complexity)"
1,ALPHA-RFR-GROUP,4.184660e+07,2.330823e+07,16293,2568.379125,99.957735,86.432798,"Top-right (High Profit, High Complexity)"
2,VGH,3.036994e+07,1.595195e+07,1440,21090.234625,99.915469,84.995773,"Top-right (High Profit, High Complexity)"



Center (Mid-range):


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
3,SONEPAR,1.391420e+07,9.100579e+06,1565,8890.860557,99.873204,59.974641,Center (Mid-range)
28,WS WEINMANN & SCHANZ GMBH,3.162147e+05,1.986525e+05,78,4054.034419,99.027895,59.847844,Center (Mid-range)
29,MÜLLER OHG,2.768674e+05,1.556527e+05,71,3899.540827,98.816568,59.678783,Center (Mid-range)



Bottom-right (Low Profit, High Complexity):


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
11,Wilhelm Layher GmbH & Co KG,3180676.85,-83057.50,30,106022.561667,39.475909,66.441251,"Bottom-right (Low Profit, High Complexity)"
17,Schwenkel & Hitzbleck GmbH,878436.22,-29951.98,25,35137.448800,39.856298,65.511412,"Bottom-right (Low Profit, High Complexity)"
27,GRUNDFOS OPERATIONS A/S,318769.68,-47105.63,63,5059.836190,39.264582,71.893491,"Bottom-right (Low Profit, High Complexity)"



Top-left (High Profit, Low Complexity):


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
46,MvD Gasarmaturen,111631.40,59229.64,12,9302.616667,98.203719,34.657650,"Top-left (High Profit, Low Complexity)"
49,Kawo Projekte GmbH,102156.66,102156.66,1,102156.660000,98.478445,0.042265,"Top-left (High Profit, Low Complexity)"
51,Watercryst Wassertechnik GmbH,98366.93,98366.93,1,98366.930000,98.351648,0.084531,"Top-left (High Profit, Low Complexity)"



Bottom-left (Low Profit, Low Complexity):


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
64,MK GEBÄUDETECHNIK GMBH,59968.010000,-128670.240108,3,19989.336667,38.165680,34.319527,"Bottom-left (Low Profit, Low Complexity)"
813,NATHAN PROJECTS B.V.,682.050000,-1459.822038,1,682.050000,35.164835,32.121724,"Bottom-left (Low Profit, Low Complexity)"
1444,BAREISS SANITÄR- UND HEIZUNGS-GROßHANDEL GMBH,76.889291,64.829291,1,76.889291,39.433643,28.951817,"Bottom-left (Low Profit, Low Complexity)"


### Filter by Quadrant

In [13]:
# Example: Show all "Top-left" customers (high profit, low complexity)
quadrant_filter = "Top-left (High Profit, Low Complexity)"
filtered = customer_matrix[customer_matrix['quadrant'] == quadrant_filter]

print(f"\n{quadrant_filter}: {len(filtered)} customer groups")
print(f"Total sales: {filtered['net_sales'].sum():,.0f}")
print(f"Total profit: {filtered['gross_profit'].sum():,.0f}")
print(f"\nDetails:")
display(filtered[display_cols])


Top-left (High Profit, Low Complexity): 613 customer groups
Total sales: 2,343,335
Total profit: 2,087,493

Details:


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
46,MvD Gasarmaturen,111631.40,59229.64,12,9302.616667,98.203719,34.657650,"Top-left (High Profit, Low Complexity)"
49,Kawo Projekte GmbH,102156.66,102156.66,1,102156.660000,98.478445,0.042265,"Top-left (High Profit, Low Complexity)"
51,Watercryst Wassertechnik GmbH,98366.93,98366.93,1,98366.930000,98.351648,0.084531,"Top-left (High Profit, Low Complexity)"
55,MvD - Gasarmaturen,78325.56,64181.07,3,26108.520000,98.140321,26.415892,"Top-left (High Profit, Low Complexity)"
66,WDT -Werner Dosiertechnik,55945.53,48348.34,9,6216.170000,97.675402,34.213863,"Top-left (High Profit, Low Complexity)"
...,...,...,...,...,...,...,...,...
991,MACHENTANZ GMBH,455.05,455.05,1,455.050000,60.312764,17.624683,"Top-left (High Profit, Low Complexity)"
992,ORBEN Wasseraufbereitung,452.50,452.50,1,452.500000,60.270499,17.666948,"Top-left (High Profit, Low Complexity)"
993,PETRA BROCKMANN,451.20,451.20,1,451.200000,60.228233,17.730347,"Top-left (High Profit, Low Complexity)"
994,FENDT HAUSTECHNIK GMBH,448.18,448.18,1,448.180000,60.185968,17.793745,"Top-left (High Profit, Low Complexity)"


### Filter by Score Ranges

In [14]:
# Example: Show high-profitability customers (>= 70th percentile) with mid-complexity (30-70)
prof_min, prof_max = 70, 100
cplx_min, cplx_max = 30, 70

filtered = customer_matrix[
    (customer_matrix['profitability_pct'] >= prof_min) & (customer_matrix['profitability_pct'] <= prof_max) &
    (customer_matrix['complexity_pct'] >= cplx_min) & (customer_matrix['complexity_pct'] <= cplx_max)
]

print(f"\nCustomer groups with:")
print(f"  Profitability: {prof_min}-{prof_max}th percentile")
print(f"  Complexity: {cplx_min}-{cplx_max}th percentile")
print(f"\nFound: {len(filtered)} customer groups")
print(f"Total sales: {filtered['net_sales'].sum():,.0f}")
print(f"\nDetails:")
display(filtered[display_cols])


Customer groups with:
  Profitability: 70-100th percentile
  Complexity: 30-70th percentile

Found: 295 customer groups
Total sales: 54,924,586

Details:


,customer_group,net_sales,gross_profit,orders,avg_order_value,profitability_pct,complexity_pct,quadrant
3,SONEPAR,1.391420e+07,9.100579e+06,1565,8890.860557,99.873204,59.974641,Center (Mid-range)
4,SOLAR,1.329114e+07,7.971148e+06,778,17083.721137,99.830938,62.848690,"Top-right (High Profit, High Complexity)"
6,PFEIFFER&MAY,8.401927e+06,5.274011e+06,3360,2500.573548,99.746407,64.370245,"Top-right (High Profit, High Complexity)"
9,NL-WASCO,3.941642e+06,2.282517e+06,714,5520.506914,99.577346,61.158073,"Top-right (High Profit, High Complexity)"
12,BUDERUS-GROUP,3.017271e+06,1.677199e+06,2661,1133.886272,99.535080,67.117498,"Top-right (High Profit, High Complexity)"
...,...,...,...,...,...,...,...,...
704,BAYWA HAUSTECHNIK GMBH,8.507300e+02,8.507300e+02,3,283.576667,71.132713,66.398986,"Top-right (High Profit, High Complexity)"
710,SWTEC SANITÄR- UND WÄRMETECHNIK GMBH,8.408500e+02,8.408500e+02,5,168.170000,70.879121,63.736264,"Top-right (High Profit, High Complexity)"
713,FLOREA HAUSTECHNIK GMBH,8.337100e+02,8.337100e+02,2,416.855000,70.794590,30.642434,"Top-left (High Profit, Low Complexity)"
716,ARNDT GMBH,8.279500e+02,8.279500e+02,2,413.975000,70.667794,30.684700,"Top-left (High Profit, Low Complexity)"


### Correlation: Drivers to Final Scores

In [15]:
# Correlation heatmap: raw drivers → final scores
corr_cols = ['net_sales', 'gross_profit', 'orders', 'avg_order_value', 'neg_line_share', 'fragmentation', 
             'profitability_pct', 'complexity_pct']
corr_matrix = customer_matrix[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, cbar_kws={'label': 'Correlation'})
ax.set_title('Correlation: Complexity & Profitability Drivers', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print("\nKey correlations with final scores:")
print(f"\nProfitability Score:")
print(corr_matrix['profitability_pct'].sort_values(ascending=False))
print(f"\nComplexity Score:")
print(corr_matrix['complexity_pct'].sort_values(ascending=False))


Key correlations with final scores:

Profitability Score:
profitability_pct    1.000000
avg_order_value      0.225407
fragmentation        0.105266
gross_profit         0.079226
net_sales            0.077066
orders               0.065102
neg_line_share      -0.495803
complexity_pct      -0.588282
Name: profitability_pct, dtype: float64

Complexity Score:
complexity_pct       1.000000
neg_line_share       0.627685
fragmentation        0.083443
orders               0.047888
net_sales            0.047686
gross_profit         0.046767
avg_order_value     -0.155278
profitability_pct   -0.588282
Name: complexity_pct, dtype: float64


C:\Users\sfr\AppData\Local\Temp\ipykernel_22128\3518324497.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Visualization: Customer Complexity × Profitability Matrix

In [16]:
# Create figure with scatter plot
fig = plt.figure(figsize=(14, 8))
ax = fig.add_axes([0.08, 0.12, 0.54, 0.72])

# Quadrant background shading
ax.axvspan(0, 40, ymin=0.6, ymax=1.0, color=LIGHT_GREEN, alpha=0.8, lw=0, zorder=0)
ax.axvspan(60, 100, ymin=0.6, ymax=1.0, color=LIGHT_BLUE, alpha=0.8, lw=0, zorder=0)
ax.axvspan(0, 40, ymin=0.0, ymax=0.4, color=LIGHT_YELLOW, alpha=0.8, lw=0, zorder=0)
ax.axvspan(60, 100, ymin=0.0, ymax=0.4, color=LIGHT_RED, alpha=0.8, lw=0, zorder=0)

# Gridlines at 40th and 60th percentiles
ax.axvline(40, color="#cccccc", lw=0.8, ls="--", alpha=0.6, zorder=1)
ax.axvline(60, color="#cccccc", lw=0.8, ls="--", alpha=0.6, zorder=1)
ax.axhline(40, color="#cccccc", lw=0.8, ls="--", alpha=0.6, zorder=1)
ax.axhline(60, color="#cccccc", lw=0.8, ls="--", alpha=0.6, zorder=1)

# Color map for quadrants
quad_colors = {
    "Top-left (High Profit, Low Complexity)": GREEN,
    "Top-right (High Profit, High Complexity)": BLUE,
    "Bottom-left (Low Profit, Low Complexity)": ORANGE,
    "Bottom-right (Low Profit, High Complexity)": RED,
    "Center (Mid-range)": GREY,
}

colors = customer_matrix['quadrant'].map(quad_colors)
sizes = (customer_matrix['net_sales'] / customer_matrix['net_sales'].max() * 600) + 50

ax.scatter(customer_matrix['complexity_pct'], customer_matrix['profitability_pct'],
          s=sizes, c=colors, alpha=0.7, edgecolors="white", lw=1.2, zorder=3)

# Label selection: top-10 by net_sales + top-3 per quadrant + outliers
labeled_idx = set()

# Top 10 by net_sales
top_10_idx = customer_matrix.nlargest(10, 'net_sales').index
labeled_idx.update(top_10_idx)

# Top 3 per quadrant
for quad in customer_matrix['quadrant'].unique():
    quad_top3 = customer_matrix[customer_matrix['quadrant'] == quad].nlargest(3, 'net_sales').index
    labeled_idx.update(quad_top3)

# Outliers
labeled_idx.add(customer_matrix['profitability_pct'].idxmax())
labeled_idx.add(customer_matrix['profitability_pct'].idxmin())
labeled_idx.add(customer_matrix['complexity_pct'].idxmax())
labeled_idx.add(customer_matrix['complexity_pct'].idxmin())

# Plot labels
for idx in labeled_idx:
    row = customer_matrix.iloc[idx]
    label = row['customer_group'][:20]
    ax.annotate(label, (row['complexity_pct'], row['profitability_pct']),
               xytext=(5, 5), textcoords='offset points',
               fontsize=7, ha='left', va='bottom',
               bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.8, linewidth=0),
               zorder=5)

# Axes
ax.set_xlabel('Complexity Percentile (0–100)\n→ Low Complexity | High Complexity →', color=BLUE, fontweight='bold')
ax.set_ylabel('Profitability Percentile (0–100)\n→ Low Profit | High Profit →', color=BLUE, fontweight='bold')
ax.set_xlim(-5, 105)
ax.set_ylim(-5, 105)
ax.set_xticks([0, 20, 40, 60, 80, 100])
ax.set_yticks([0, 20, 40, 60, 80, 100])

# Quadrant labels
qkw = dict(fontsize=9, fontweight='bold', color='#333333', zorder=5)
ax.text(20, 80, "Grow\n(Valuable,\nEasy)", ha='center', va='center', **qkw)
ax.text(80, 80, "Protect /\nServe Differently\n(Valuable,\nCostly)", ha='center', va='center', **qkw)
ax.text(20, 20, "Simplify / Steer\n(Marginal,\nEasy)", ha='center', va='center', **qkw)
ax.text(80, 20, "Selective\nDeprioritization\n(Marginal,\nCostly)", ha='center', va='center', **qkw)

# Title and footer
fig.text(0.012, 0.965, "2 Approach", fontsize=8, color="#555555")
fig.text(0.98, 0.965, "confidential", fontsize=8, color="#555555", ha="right")
fig.text(0.012, 0.93, "Customer Groups: Complexity & Profitability Analysis",
        fontsize=13, fontweight="bold", color=BLUE)
fig.text(0.012, 0.895,
        "Each point = a customer group. Position indicates profitability (Y) and complexity (X).",
        fontsize=9.5, color="#333333")

# Comment box
quad_dist = customer_matrix['quadrant'].value_counts().to_dict()
lines = [
    f"• {len(customer_matrix)} customer groups analyzed.",
    "",
    f"• Distribution across quadrants:",
    f"  - High profit, low complexity: {quad_dist.get('Top-left (High Profit, Low Complexity)', 0)}",
    f"  - High profit, high complexity: {quad_dist.get('Top-right (High Profit, High Complexity)', 0)}",
    f"  - Low profit, low complexity: {quad_dist.get('Bottom-left (Low Profit, Low Complexity)', 0)}",
    f"  - Low profit, high complexity: {quad_dist.get('Bottom-right (Low Profit, High Complexity)', 0)}",
    f"  - Mid-range: {quad_dist.get('Center (Mid-range)', 0)}",
    "",
]

for quad in ["Top-left (High Profit, Low Complexity)", "Top-right (High Profit, High Complexity)",
             "Bottom-right (Low Profit, High Complexity)"]:
    quad_cg = customer_matrix[customer_matrix['quadrant'] == quad].nlargest(2, 'net_sales')
    if not quad_cg.empty:
        short_quad = quad.split("(")[0].strip()
        lines.append(f"• {short_quad}: {quad_cg.iloc[0]['customer_group']}")

lines += [
    "",
    "Profitability = 60% gross profit + 40% sales volume.",
    "Complexity = order frequency (30%) + small-order",
    "burden (30%) + margin leakage (20%) +",
    "fragmentation (20%); each is percentile-ranked.",
]

fig.text(0.67, 0.84, "Comments", fontsize=9.5, fontweight='bold', color=BLUE)
fig.text(0.67, 0.82, "\n".join(lines), fontsize=7.5, color="#222222", va='top')

# Footer
fig.text(0.012, 0.015,
        "Source: GF BFS silver layer (customer.parquet) | "
        "Helbling proof-of-concept | numbers deterministic",
        fontsize=6.5, color="#888888")
fig.text(0.98, 0.015, "GF_BFS_CustomerMatrix", fontsize=6.5, color="#888888", ha="right")

plt.show()
print("Chart displayed.")

Chart displayed.


C:\Users\sfr\AppData\Local\Temp\ipykernel_22128\3456146853.py:123: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Optional: Distribution Plots

In [17]:
# Distributions of individual complexity drivers and final scores
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Complexity & Profitability Score Distributions', fontsize=14, fontweight='bold')

# Order frequency
axes[0, 0].hist(customer_matrix['orders'], bins=30, color=BLUE, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Order Count')
axes[0, 0].set_ylabel('Customer Groups')
axes[0, 0].set_title('Order Frequency Distribution')

# Average order value
axes[0, 1].hist(customer_matrix['avg_order_value'], bins=30, color=ORANGE, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Average Order Value')
axes[0, 1].set_ylabel('Customer Groups')
axes[0, 1].set_title('Average Order Value Distribution')

# Margin leakage
axes[0, 2].hist(customer_matrix['neg_line_share'], bins=30, color=RED, alpha=0.7, edgecolor='black')
axes[0, 2].set_xlabel('Negative Line Share')
axes[0, 2].set_ylabel('Customer Groups')
axes[0, 2].set_title('Margin Leakage Distribution')

# Profitability percentile
axes[1, 0].hist(customer_matrix['profitability_pct'], bins=30, color=GREEN, alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Profitability Percentile (0-100)')
axes[1, 0].set_ylabel('Customer Groups')
axes[1, 0].set_title('Profitability Score Distribution')
axes[1, 0].axvline(60, color='red', linestyle='--', linewidth=2, label='60th percentile')
axes[1, 0].legend()

# Complexity percentile
axes[1, 1].hist(customer_matrix['complexity_pct'], bins=30, color=LBLUE, alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Complexity Percentile (0-100)')
axes[1, 1].set_ylabel('Customer Groups')
axes[1, 1].set_title('Complexity Score Distribution')
axes[1, 1].axvline(40, color='orange', linestyle='--', linewidth=2, label='40th percentile')
axes[1, 1].axvline(60, color='red', linestyle='--', linewidth=2, label='60th percentile')
axes[1, 1].legend()

# Fragmentation
axes[1, 2].hist(customer_matrix['fragmentation'], bins=30, color=GREY, alpha=0.7, edgecolor='black')
axes[1, 2].set_xlabel('Fragmentation (Regions + Sales Units)')
axes[1, 2].set_ylabel('Customer Groups')
axes[1, 2].set_title('Fragmentation Distribution')

plt.tight_layout()
plt.show()
print("Distribution plots displayed.")

Distribution plots displayed.


C:\Users\sfr\AppData\Local\Temp\ipykernel_22128\1436440937.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Export Results

In [18]:
# Select columns for export
export_cols = [
    'customer_group',
    'net_sales',
    'gross_profit',
    'orders',
    'avg_order_value',
    'neg_line_share',
    'regions',
    'sales_units',
    'fragmentation',
    'gp_pct',
    'value_raw',
    'profitability_pct',
    'complexity_raw',
    'complexity_pct',
    'quadrant',
]

export_data = customer_matrix[export_cols].copy()
export_data = export_data.rename(columns={'customer_group': 'Customer Group'})

print(f"Export data shape: {export_data.shape}")
print(f"\nFirst 10 rows:")
display(export_data.head(10))

Export data shape: (2366, 15)

First 10 rows:


,Customer Group,net_sales,gross_profit,orders,avg_order_value,neg_line_share,regions,sales_units,fragmentation,gp_pct,value_raw,profitability_pct,complexity_raw,complexity_pct,quadrant
0,GC,1.072299e+08,5.880533e+07,45613,2350.862339,0.086727,2,3,5,54.840432,100.000000,100.000000,65.942519,86.348267,"Top-right (High Profit, High Complexity)"
1,ALPHA-RFR-GROUP,4.184660e+07,2.330823e+07,16293,2568.379125,0.162639,1,2,3,55.699219,99.957735,99.957735,66.115807,86.432798,"Top-right (High Profit, High Complexity)"
2,VGH,3.036994e+07,1.595195e+07,1440,21090.234625,0.146135,2,2,4,52.525470,99.915469,99.915469,64.750634,84.995773,"Top-right (High Profit, High Complexity)"
3,SONEPAR,1.391420e+07,9.100579e+06,1565,8890.860557,0.032037,1,1,2,65.404991,99.873204,99.873204,54.040575,59.974641,Center (Mid-range)
4,SOLAR,1.329114e+07,7.971148e+06,778,17083.721137,0.152560,1,1,2,59.973418,99.830938,99.830938,54.761200,62.848690,"Top-right (High Profit, High Complexity)"
5,EDT,9.479942e+06,5.717621e+06,2764,3429.790968,0.102564,2,3,5,60.312824,99.788673,99.788673,65.321217,85.587489,"Top-right (High Profit, High Complexity)"
6,PFEIFFER&MAY,8.401927e+06,5.274011e+06,3360,2500.573548,0.081719,1,1,2,62.771449,99.746407,99.746407,55.663567,64.370245,"Top-right (High Profit, High Complexity)"
7,GSH,5.354182e+06,3.459986e+06,3186,1680.534259,0.060456,1,2,3,64.622113,99.704142,99.704142,66.606086,86.939983,"Top-right (High Profit, High Complexity)"
8,WIM,4.878231e+06,2.825616e+06,2199,2218.385931,0.102291,1,2,3,57.922967,99.661877,99.661877,66.052409,86.390533,"Top-right (High Profit, High Complexity)"
9,NL-WASCO,3.941642e+06,2.282517e+06,714,5520.506914,0.043423,1,1,2,57.907760,99.594252,99.577346,54.353339,61.158073,"Top-right (High Profit, High Complexity)"


In [ ]:
# Save to CSV
csv_path = EXPORT_DIR / 'customer_matrix_results.csv'
export_data.to_csv(csv_path, index=False)
print(f"Data saved to CSV: {csv_path}")

# Save to XLSX
xlsx_path = EXPORT_DIR / 'customer_matrix_results.xlsx'
export_data.to_excel(xlsx_path, index=False, sheet_name='Customer Matrix')
print(f"Data saved to XLSX: {xlsx_path}")

print(f"\n✓ Export complete")

## Conclusion

This notebook successfully:

1. ✓ Loaded customer transaction data from **silver layer** (raw, unaggregate data)
2. ✓ Aggregated by Customer Group and computed summary metrics
3. ✓ Computed **Profitability Score** (Y-axis) = 60% gross profit + 40% sales, percentile-ranked 0-100
4. ✓ Computed **Complexity Score** (X-axis) = weighted combination of:
   - Order frequency (30%)
   - Small order burden (30%)
   - Margin leakage (20%)
   - Fragmentation (20%)
5. ✓ Classified customer groups into 5 strategic quadrants based on 60/40 percentile thresholds
6. ✓ Enabled interactive exploration:
   - Filter by quadrant
   - Filter by score ranges
   - View top customers globally and per quadrant
   - Analyze correlation between drivers and final scores
7. ✓ Visualized the 2D scatter matrix with quadrant shading and bubble sizing
8. ✓ Generated optional distribution plots for individual drivers
9. ✓ Exported underlying data to CSV and XLSX

**Matrix Key Concept:** The chart reveals strategic customer portfolio management opportunities. Customer groups in the **top-left** (high profit, low complexity) deserve growth investment, while those in the **bottom-right** (low profit, high complexity) may require deprioritization or restructuring.